In [ ]:
# Updated 2026-11-02. Paste the S3 object URIs used by your completed inference Postprocess job.
# Use the job's actual input locations when upstream steps were cached.
MODEL_ID = "usecase2"
S3_INPUTS = {
    "data": "",             # GetData: steps/GetData/raw/data.csv (raw, not prepared).
    "selection": "",        # Preprocess: steps/Preprocess/selection/selected_features.csv.
    "model": "",            # steps/ResolveModel/package.json: checkpoint (the training model.tar.gz).
    "predictions": "",      # Score: steps/Score/predictions.csv.
    "messaging": "",        # Score: steps/Score/messaging.csv.
    "messaging_final": "",  # Score: steps/Score/messaging_final.csv.
}


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import os
import sys
import tarfile

import boto3
import pandas as pd
from botocore.config import Config as ClientConfig
from IPython import get_ipython
from IPython.display import display

# Reload edited model functions too when rerunning the final cell.
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

# Run the Python code in this checkout, not code stored inside the model archive.
REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))

# Rerun setup for fresh inputs; rerun the final cell to debug without downloading again.
work_directory = Path(mkdtemp(prefix="postprocess-inference-"))
data_directory = work_directory / "raw"
model_directory = work_directory / "model"
output_directory = work_directory / "output"
output_directory.mkdir()
selection_file = work_directory / "selection/selected_features.csv"
prediction_directory = work_directory / "predictions"
local_files = {
    "data": data_directory / "data.csv",
    "selection": selection_file,
    "model": work_directory / "model.tar.gz",
    "predictions": prediction_directory / "predictions.csv",
    "messaging": prediction_directory / "messaging.csv",
    "messaging_final": prediction_directory / "messaging_final.csv",
}

# Set these before importing the entrypoint or the model code.
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_ANALYSIS_MODE": "prediction_inference",
    "MLOPS_MODEL_CODE_SOURCE": "pipeline",
    "MLOPS_MODEL_ARTIFACT_URI": S3_INPUTS["model"],
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})

# Copy these exact S3 objects unchanged into the local input folders.
s3 = boto3.client("s3", config=ClientConfig(
    connect_timeout=5, read_timeout=15, retries={"total_max_attempts": 2}))
for name, local_file in local_files.items():
    address = urlsplit(S3_INPUTS[name])
    if address.scheme != "s3" or not address.netloc or not address.path.strip("/"):
        raise ValueError(f"Set S3_INPUTS[{name!r}] to an S3 object path in the first cell.")
    local_file.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {S3_INPUTS[name]} -> {local_file}", flush=True)
    s3.download_file(address.netloc, address.path.lstrip("/"), str(local_file))

# model_directory is the extracted model root, not the archive or its parent.
with tarfile.open(local_files["model"]) as archive:
    archive.extractall(model_directory, filter="data")

print("Model directory:", model_directory)
print("Saved model parameters:", model_directory / "parameters.json")
print("Saved model metadata:", model_directory / "model_metadata.json")
print("Local output directory:", output_directory)


In [ ]:
# Optional checks/inspection; the entrypoint call does not depend on this cell.
for local_file in local_files.values():
    assert local_file.is_file(), f"Missing input: {local_file}"
assert (model_directory / "parameters.json").is_file(), "Expected parameters.json at the extracted model root."
# Preprocess copies this file unchanged from the model archive.
assert selection_file.read_bytes() == (model_directory / "selected_features.csv").read_bytes(), "Selection and model archive do not match."
display(pd.read_csv(data_directory / "data.csv", nrows=10, dtype=str))
display(pd.read_csv(selection_file))
display(pd.read_csv(prediction_directory / "messaging_final.csv", nrows=10, dtype=str))


In [ ]:
from mlops_models.entrypoints import postprocess_inference

postprocess_inference = importlib.reload(postprocess_inference)
print("Entrypoint:", postprocess_inference.__file__)
result_directory = postprocess_inference.execute(
    model_id=MODEL_ID,
    input_path=prediction_directory,
    output_dir=output_directory,
    model_dir=model_directory,
    data_directory=data_directory,
    selection_file=selection_file,
)
print("Outputs:", output_directory)
result_directory
